# Matrix Decompositions: LU, QR, and SVD

A **decomposition** writes one matrix as a product of simpler ones (triangular, orthogonal, diagonal). Throughout, $A$ is $n \times k$ ($n$ rows, $k$ columns), square or not.

| | size of $A$ | form | sizes of the factors | built from |
|---|---|---|---|---|
| **LU** | $n\times k$, any | $A = LU$ | $L$: $n\times n$, $U$: $n\times k$ | Gaussian elimination |
| **QR** | $n\times k$, $n \ge k$ | $A = QR$ | $Q$: $n\times k$, $R$: $k\times k$ | Gram–Schmidt |
| **SVD** | $n\times k$, any | $A = U\Sigma V^T$ | $U$: $n\times n$, $\Sigma$: $n\times k$, $V$: $k\times k$ | eigenvalues of $A^TA$ |

In [79]:
import sympy as sp
import numpy as np
import scipy.linalg as sla        # NumPy has no LU; SciPy does
from IPython.display import display

**Matrix product: use `@`.**

| | SymPy | NumPy |
|---|---|---|
| `A @ B` | matrix product | matrix product |
| `A * B` | matrix product | **entry-by-entry** product |

To avoid mistakes, this notebook uses `@` everywhere.

In [80]:
X = np.array([[1, 2], [3, 4]])
print(X @ X)     # matrix product
print(X * X)     # entry by entry: NOT the matrix product

[[ 7 10]
 [15 22]]
[[ 1  4]
 [ 9 16]]


## Part A. LU decomposition

$$ \underset{n\times k}{A} = \underset{n\times n}{L}\ \underset{n\times k}{U}, \qquad L = \text{lower triangular with 1s on the diagonal}, \quad U = \text{upper triangular (row echelon form)}. $$

$U$ is the result of Gaussian elimination. $L$ records the **multipliers** used: entry $\ell_{ij}$ is the multiple of row $j$ subtracted from row $i$.

**Example (square, $3\times 3$).**
$$ A = \begin{pmatrix} 2 & 1 & 1 \\ 4 & 3 & 3 \\ 8 & 7 & 9 \end{pmatrix}. $$
Subtract $2\times$ row 1 from row 2 and $4\times$ row 1 from row 3, then $3\times$ (new) row 2 from row 3. The multipliers $2, 4, 3$ go into $L$.

In SymPy: `L, U, perm = A.LUdecomposition()`. Here `perm` lists any row swaps (empty means none).

In [81]:
A = sp.Matrix([[2, 1, 1],
               [4, 3, 3],
               [8, 7, 9]])
L, U, perm = A.LUdecomposition()
display(L, U)
print("row swaps:", perm)
print("L @ U == A ?", L @ U == A)

Matrix([
[1, 0, 0],
[2, 1, 0],
[4, 3, 1]])

Matrix([
[2, 1, 1],
[0, 1, 1],
[0, 0, 2]])

row swaps: []
L @ U == A ? True


**Determinant from LU** (square $A$). $\det L = 1$ (triangular, 1s on the diagonal), so
$$ \det A = \det U = \text{product of the diagonal of } U. $$
Here $2 \cdot 1 \cdot 2 = 4$.

In [82]:
print("product of diag(U):", sp.prod(U.diagonal()))
print("det A:             ", A.det())

product of diag(U): 4
det A:              4


**Non-square example ($3 \times 2$).** Elimination works the same way. $L$ is still $n\times n = 3\times 3$; $U$ has the shape of $A$, $3\times 2$.

In [83]:
A = sp.Matrix([[1, 2],
               [3, 4],
               [5, 6]])
L, U, perm = A.LUdecomposition()
display(L, U)
print("shapes: A", A.shape, "  L", L.shape, "  U", U.shape)
print("L @ U == A ?", L @ U == A)

Matrix([
[1, 0, 0],
[3, 1, 0],
[5, 2, 1]])

Matrix([
[1,  2],
[0, -2],
[0,  0]])

shapes: A (3, 2)   L (3, 3)   U (3, 2)
L @ U == A ? True


### Row echelon form and RREF

$U$ is a **row echelon form** of $A$. Going further (scale each pivot to $1$, clear the entries above each pivot) gives the **reduced row echelon form** (RREF).

- SymPy: `R, pivots = A.rref()` returns the RREF and the indices of the pivot columns.
- NumPy has no `rref`. Numerically, use the $U$ from `sla.lu(A)` as the echelon form. (SciPy's $U$ is $\min(n,k)\times k$, so for tall $A$ the zero rows at the bottom are left out.)

In [84]:
R, pivots = A.rref()
display(R)
print("pivot columns:", pivots)

Pn, Ln, Un = sla.lu(np.array(A, dtype=float))
print("echelon form from SciPy's LU:\n", Un)

Matrix([
[1, 0],
[0, 1],
[0, 0]])

pivot columns: (0, 1)
echelon form from SciPy's LU:
 [[5.  6. ]
 [0.  0.8]]


**Caution:** when the columns are **dependent**, SciPy's $U$ is triangular but may not be a true echelon form (two rows can have their first nonzero in the same column, and tiny round-off values can hide zeros). Use SymPy's `rref()` when you need the exact pivot structure.

$$ C = \begin{pmatrix} 1 & 2 & 3 \\ 2 & 4 & 7 \\ 1 & 2 & 4 \end{pmatrix} \quad (\text{column 2} = 2 \times \text{column 1}). $$

In [85]:
C = sp.Matrix([[1, 2, 3],
               [2, 4, 7],
               [1, 2, 4]])
R, pivots = C.rref()
display(R)
print("pivot columns:", pivots)

Pn, Ln, Un = sla.lu(np.array(C, dtype=float))
print("SciPy U:\n", Un)     # rows 2 and 3 both start in column 3

Matrix([
[1, 2, 0],
[0, 0, 1],
[0, 0, 0]])

pivot columns: (0, 2)
SciPy U:
 [[ 2.   4.   7. ]
 [ 0.   0.  -0.5]
 [ 0.   0.   0.5]]


### Pivoting: when a row swap is needed

If a pivot is $0$, elimination stalls and we must swap rows. Then the factorization is
$$ PA = LU, $$
where $P$ ($n\times n$) is a **permutation matrix** (the identity with its rows swapped).

$$ A_2 = \begin{pmatrix} 0 & 2 & 1 \\ 1 & 1 & 1 \\ 2 & 1 & 3 \end{pmatrix} \quad \text{has } a_{11} = 0. $$

In [86]:
A2 = sp.Matrix([[0, 2, 1],
                [1, 1, 1],
                [2, 1, 3]])
L2, U2, perm2 = A2.LUdecomposition()
print("row swaps:", perm2)                 # [[0, 1]] = swap rows 0 and 1
P2 = sp.eye(3).permute_rows(perm2)        # build P from the swaps
display(P2, L2, U2)
print("P @ A == L @ U ?", P2 @ A2 == L2 @ U2)

row swaps: [[0, 1]]


Matrix([
[0, 1, 0],
[1, 0, 0],
[0, 0, 1]])

Matrix([
[1,    0, 0],
[0,    1, 0],
[2, -1/2, 1]])

Matrix([
[1, 1,   1],
[0, 2,   1],
[0, 0, 3/2]])

P @ A == L @ U ? True


**Determinant with swaps.** Each swap flips the sign: $\det A = (-1)^{\#\text{swaps}} \cdot \prod \operatorname{diag}(U)$. Here one swap, so $\det A_2 = -(1 \cdot 2 \cdot \tfrac32) = -3$.

In [87]:
print("(-1)^swaps * prod diag(U):", (-1)**len(perm2) * sp.prod(U2.diagonal()))
print("det A2:                    ", A2.det())

(-1)^swaps * prod diag(U): -3
det A2:                     -3


**NumPy/SciPy:** `P, L, U = sla.lu(A)`. Three differences from SymPy:

1. SciPy's convention is $A = PLU$ (the $P$ is on the **other side**, so it is the transpose of the $P$ above).
2. SciPy **always** swaps to put the largest pivot on top (*partial pivoting*, better for rounding error). So its $L$ and $U$ can differ from SymPy's. Both are correct.
3. For non-square $A$, SciPy returns the **reduced** sizes: with $m = \min(n, k)$, $L$ is $n\times m$ and $U$ is $m\times k$.

In [88]:
Pn, Ln, Un = sla.lu(np.array(A2, dtype=float))
print(Pn); print(Ln); print(Un)
print("P @ L @ U == A2 ?", np.allclose(Pn @ Ln @ Un, np.array(A2, dtype=float)))

An = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)      # 3 x 2
Pn, Ln, Un = sla.lu(An)
print("\n3x2 shapes: P", Pn.shape, "  L", Ln.shape, "  U", Un.shape)
print("P @ L @ U == A ?", np.allclose(Pn @ Ln @ Un, An))

[[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
[[1.   0.   0.  ]
 [0.   1.   0.  ]
 [0.5  0.25 1.  ]]
[[ 2.    1.    3.  ]
 [ 0.    2.    1.  ]
 [ 0.    0.   -0.75]]
P @ L @ U == A2 ? True

3x2 shapes: P (3, 3)   L (3, 2)   U (2, 2)
P @ L @ U == A ? True


## Part B. QR decomposition

For $A$ of size $n\times k$ with $n \ge k$ and independent columns:
$$ \underset{n\times k}{A} = \underset{n\times k}{Q}\ \underset{k\times k}{R}, \qquad Q^TQ = I_k \ \ (\text{orthonormal columns}), \quad R = \text{upper triangular}. $$

The columns of $Q$ come from running **Gram–Schmidt** on the columns of $A$. $R = Q^TA$ records the coefficients.

**Example ($3\times 2$).**
$$ A = \begin{pmatrix} 1 & 3 \\ 2 & 3 \\ 2 & 0 \end{pmatrix}. $$
Column 1 has length $3$, so $\mathbf{q}_1 = \tfrac13(1, 2, 2)$. Remove the $\mathbf{q}_1$-part from column 2: $(3,3,0) - 3\,\mathbf{q}_1 = (2, 1, -2)$, which also has length $3$, so $\mathbf{q}_2 = \tfrac13(2, 1, -2)$.

In [89]:
A = sp.Matrix([[1, 3],
               [2, 3],
               [2, 0]])
Q, R = A.QRdecomposition()
display(Q, R)
print("shapes: A", A.shape, "  Q", Q.shape, "  R", R.shape)
print("Q^T Q == I_2 ?", Q.T @ Q == sp.eye(2))
print("Q @ R == A ?  ", Q @ R == A)

Matrix([
[1/3,  2/3],
[2/3,  1/3],
[2/3, -2/3]])

Matrix([
[3, 3],
[0, 3]])

shapes: A (3, 2)   Q (3, 2)   R (2, 2)
Q^T Q == I_2 ? True
Q @ R == A ?   True


**Careful:** $Q^TQ = I_k$, but $QQ^T \neq I_n$ when $n > k$. A $3\times 2$ matrix $Q$ has only $2$ orthonormal columns; they cannot fill all of $\mathbb{R}^3$.

In [90]:
display(Q @ Q.T)

Matrix([
[ 5/9, 4/9, -2/9],
[ 4/9, 5/9,  2/9],
[-2/9, 2/9,  8/9]])

**Reduced vs complete QR.** The version above is the **reduced** QR. The **complete** QR adds $n - k$ more orthonormal columns to $Q$:

| | $Q$ | $R$ |
|---|---|---|
| reduced (default) | $n\times k$ | $k\times k$ |
| complete | $n\times n$ (square, orthogonal) | $n\times k$ (extra rows are $0$) |

**NumPy:** `Q, R = np.linalg.qr(A)` (reduced), or `np.linalg.qr(A, mode="complete")`.

**Pitfall:** NumPy's columns of $Q$ (and rows of $R$) may have the **opposite sign** from SymPy's. That is fine: flipping $\mathbf{q}_j \to -\mathbf{q}_j$ and row $j$ of $R$ together leaves $QR$ unchanged. Always check with `np.allclose(Q @ R, A)`, not by comparing $Q$ entry by entry.

In [91]:
An = np.array([[1, 3], [2, 3], [2, 0]], dtype=float)

Qn, Rn = np.linalg.qr(An)
print("reduced:  Q", Qn.shape, " R", Rn.shape)
print(Qn); print(Rn)
print("Q @ R == A ?", np.allclose(Qn @ Rn, An))

Qc, Rc = np.linalg.qr(An, mode="complete")
print("\ncomplete: Q", Qc.shape, " R", Rc.shape)
print(Qc); print(Rc)
print("Q^T Q == I_3 ?", np.allclose(Qc.T @ Qc, np.eye(3)), "   Q @ R == A ?", np.allclose(Qc @ Rc, An))

reduced:  Q (3, 2)  R (2, 2)
[[-0.33333333 -0.66666667]
 [-0.66666667 -0.33333333]
 [-0.66666667  0.66666667]]
[[-3. -3.]
 [ 0. -3.]]
Q @ R == A ? True

complete: Q (3, 3)  R (3, 2)
[[-0.33333333 -0.66666667 -0.66666667]
 [-0.66666667 -0.33333333  0.66666667]
 [-0.66666667  0.66666667 -0.33333333]]
[[-3. -3.]
 [ 0. -3.]
 [ 0.  0.]]
Q^T Q == I_3 ? True    Q @ R == A ? True


**Determinant from QR** (square $A$, $n = k$). Now $Q$ is square with $Q^TQ = I$, so $(\det Q)^2 = 1$ and $\det Q = \pm 1$. Therefore
$$ |\det A| = |\det R| = \text{product of the diagonal of } R. $$

In [92]:
B = sp.Matrix([[2, 0, 1],
               [1, 3, 2],
               [1, 1, 2]])
QB, RB = B.QRdecomposition()
print("product of diag(R):", sp.simplify(sp.prod(RB.diagonal())))
print("det B:             ", B.det())

product of diag(R): 6
det B:              6


## Part C. Singular value decomposition (SVD)

The SVD works for **every** $n \times k$ matrix:
$$ \underset{n\times k}{A} = \underset{n\times n}{U}\ \underset{n\times k}{\Sigma}\ \underset{k\times k}{V^T}, $$
- $U$ and $V$ are **orthogonal**: $U^TU = I_n$, $V^TV = I_k$,
- $\Sigma$ is "diagonal" (zero except $\sigma_{11}, \sigma_{22}, \dots$) with entries $\sigma_1 \ge \sigma_2 \ge \dots \ge 0$, the **singular values**. There are $\min(n, k)$ of them.

**How to find the singular values:** $A^TA$ is a $k\times k$ symmetric matrix. Its **eigenvalues** are the numbers $\lambda$ with
$$ \det(A^TA - \lambda I) = 0, $$
and they are all $\ge 0$. Then
$$ \sigma_i = \sqrt{\lambda_i}. $$

**Example ($2\times 3$).**
$$ A = \begin{pmatrix} 3 & 2 & 2 \\ 2 & 3 & -2 \end{pmatrix}. $$

In [93]:
A = sp.Matrix([[3, 2,  2],
               [2, 3, -2]])
AtA = A.T @ A
display(AtA)

lam = sp.symbols("lambda")
p = (AtA - lam*sp.eye(3)).det()
print("det(A^T A - lambda I) =", sp.factor(p))
print("roots:", sp.solve(p, lam))
print("same with eigenvals():", AtA.eigenvals())     # {eigenvalue: multiplicity}
print("singular values:      ", A.singular_values())

Matrix([
[13, 12,  2],
[12, 13, -2],
[ 2, -2,  8]])

det(A^T A - lambda I) = -lambda*(lambda - 25)*(lambda - 9)
roots: [0, 9, 25]
same with eigenvals(): {25: 1, 9: 1, 0: 1}
singular values:       [5, 3, 0]


The roots $25, 9, 0$ give $\sigma_1 = 5$, $\sigma_2 = 3$. The third root $0$ comes from $A^TA$ being $3\times 3$ while $A$ has only $\min(2, 3) = 2$ singular values. SymPy still lists it as $0$; NumPy returns only $\min(n, k)$ values.

**NumPy is the usual tool for the SVD:** `U, s, Vt = np.linalg.svd(A)`. Note it returns:
- `s` as a **1-D array** of singular values (largest first), not the matrix $\Sigma$,
- `Vt` $= V^T$, **already transposed**.

In [94]:
An = np.array([[3, 2, 2], [2, 3, -2]], dtype=float)
U, s, Vt = np.linalg.svd(An)
print("shapes: U", U.shape, "  s", s.shape, "  Vt", Vt.shape)
print("s =", s)

Sigma = np.zeros(An.shape)               # rebuild the n x k Sigma
Sigma[:len(s), :len(s)] = np.diag(s)
print("Sigma =\n", Sigma)
print("U @ Sigma @ Vt == A ?", np.allclose(U @ Sigma @ Vt, An))
print("U orthogonal?", np.allclose(U.T @ U, np.eye(2)), "  V orthogonal?", np.allclose(Vt @ Vt.T, np.eye(3)))

shapes: U (2, 2)   s (2,)   Vt (3, 3)
s = [5. 3.]
Sigma =
 [[5. 0. 0.]
 [0. 3. 0.]]
U @ Sigma @ Vt == A ? True
U orthogonal? True   V orthogonal? True


**Reduced SVD.** Only the first $m = \min(n, k)$ columns of $U$ and $V$ ever meet a nonzero $\sigma$. Dropping the rest gives
$$ \underset{n\times k}{A} = \underset{n\times m}{U}\ \underset{m\times m}{\Sigma}\ \underset{m\times k}{V^T}, $$
with $\Sigma$ now a square diagonal matrix. NumPy: `np.linalg.svd(A, full_matrices=False)`.

In [95]:
U, s, Vt = np.linalg.svd(An, full_matrices=False)
print("shapes: U", U.shape, "  Sigma", np.diag(s).shape, "  Vt", Vt.shape)
print("U @ diag(s) @ Vt == A ?", np.allclose(U @ np.diag(s) @ Vt, An))

shapes: U (2, 2)   Sigma (2, 2)   Vt (2, 3)
U @ diag(s) @ Vt == A ? True


**SymPy** has `U, S, V = A.singular_value_decomposition()` with $A = USV^T$. It returns a reduced form keeping only the **nonzero** $\sigma$'s, listed **smallest first**. Exact SVDs get messy quickly, so use SymPy mainly for `singular_values()`.

In [96]:
Us, Ss, Vs = A.singular_value_decomposition()
display(Ss)
print("U @ S @ V^T == A ?", sp.simplify(Us @ Ss @ Vs.T) == A)

Matrix([
[3, 0],
[0, 5]])

U @ S @ V^T == A ? True


**What the SVD tells you.**

- **Rank** = number of nonzero singular values. (Use a tolerance in NumPy: `np.sum(s > 1e-10)`.)
- **Determinant** (square $A$): $\det U, \det V = \pm 1$, so $|\det A| = \sigma_1 \sigma_2 \cdots \sigma_n$.
- **Geometry:** $A$ maps the unit circle/sphere to an ellipse/ellipsoid with semi-axes $\sigma_1, \sigma_2, \dots$. So $|\det A|$, the area/volume scale factor, is the product of the semi-axes.

In [97]:
Bn = np.array([[2, 0, 1], [1, 3, 2], [1, 1, 2]], dtype=float)
s = np.linalg.svd(Bn, compute_uv=False)     # singular values only
print("singular values:", s)
print("product:", np.prod(s), "   det:", np.linalg.det(Bn))

Sn = np.array([[1, 2, 3], [4, 5, 6], [5, 7, 9]], dtype=float)   # row 3 = row 1 + row 2
s = np.linalg.svd(Sn, compute_uv=False)
print("\nsingular values:", s, "  rank:", np.sum(s > 1e-10))

singular values: [4.56783625 1.91415026 0.68622209]
product: 5.999999999999998    det: 6.0

singular values: [1.56633231e+01 8.12593979e-01 1.47416704e-15]   rank: 2


### Summary

$A$ is $n\times k$, $m = \min(n, k)$.

| | form | sizes | needs | $\det A$ (square) |
|---|---|---|---|---|
| **LU** | $PA = LU$ | $P, L$: $n\times n$; $U$: $n\times k$ | any | $\pm\prod \operatorname{diag} U$ |
| **QR** (reduced) | $A = QR$ | $Q$: $n\times k$; $R$: $k\times k$ | $n \ge k$, indep. columns | $\pm\prod \operatorname{diag} R$ |
| **QR** (complete) | $A = QR$ | $Q$: $n\times n$; $R$: $n\times k$ | $n \ge k$ | |
| **SVD** (full) | $A = U\Sigma V^T$ | $U$: $n\times n$; $\Sigma$: $n\times k$; $V$: $k\times k$ | **any** | $\pm\prod \sigma_i$ |
| **SVD** (reduced) | $A = U\Sigma V^T$ | $U$: $n\times m$; $\Sigma$: $m\times m$; $V$: $k\times m$ | **any** | |

## Exercises

Solve with **SymPy** (exact); check with **NumPy/SciPy** where asked. Use `@` for matrix products.

**Syntax you will need:**

| task | SymPy | NumPy / SciPy |
|---|---|---|
| make a matrix | `sp.Matrix([[1, 2], [3, 4]])` | `np.array([[1, 2], [3, 4]], dtype=float)` |
| matrix from a rule $a_{ij} = f(i, j)$ | `sp.Matrix(n, k, lambda i, j: ...)` | `np.fromfunction(lambda i, j: ..., (n, k))` |
| $\min(i, j)$, $\max(i, j)$ inside a rule | `min(i, j)`, `max(i, j)` | `np.minimum(i, j)`, `np.maximum(i, j)` |
| convert SymPy → NumPy | | `np.array(A, dtype=float)` |
| symbol | `t = sp.symbols("t")` | |
| substitute | `A.subs(t, 0)` | |
| matrix product | `A @ B` | `A @ B` |
| transpose | `A.T` | `A.T` |
| shape | `A.shape` | `A.shape` |
| identity | `sp.eye(n)` | `np.eye(n)` |
| determinant | `A.det()` | `np.linalg.det(A)` |
| RREF | `R, pivots = A.rref()` | none; echelon form = `U` from `sla.lu(A)` |
| LU | `L, U, perm = A.LUdecomposition()` | `P, L, U = sla.lu(A)` ($A = PLU$) |
| permutation matrix from swaps | `sp.eye(n).permute_rows(perm)` | |
| QR (reduced) | `Q, R = A.QRdecomposition()` | `Q, R = np.linalg.qr(A)` |
| QR (complete) | | `np.linalg.qr(A, mode="complete")` |
| diagonal entries | `A.diagonal()` | `np.diag(A)` |
| product of entries | `sp.prod(...)` | `np.prod(...)` |
| eigenvalues | `A.eigenvals()` | |
| singular values | `A.singular_values()` | `np.linalg.svd(A, compute_uv=False)` |
| full SVD | | `U, s, Vt = np.linalg.svd(A)` |
| reduced SVD | | `np.linalg.svd(A, full_matrices=False)` |
| simplify | `sp.simplify(expr)` | |
| equal up to rounding | | `np.allclose(X, Y)` |

**Exercise 1 (LU).** Let $M$ be the $4\times 4$ matrix with entries $m_{ij} = \min(i, j) + 1$, $\ i, j = 0, 1, 2, 3$.

(a) Build $M$ from the rule in **SymPy** and in **NumPy** (`np.fromfunction`). Find its RREF with SymPy.

(b) Compute the LU decomposition in **SymPy** and in **SciPy**. Check $LU = M$ and $PLU = M$.

(c) Find $\det M$ from the diagonal of $U$ in both. Confirm with `M.det()`.

*Hint:* in SymPy use Python's `min(i, j)`. In `np.fromfunction`, `i` and `j` are whole arrays, so use `np.minimum(i, j)` instead (plain `min` gives an error).

In [98]:
# Exercise 1

**Exercise 2 (QR).** Let
$$ A = \begin{pmatrix} 2 & 5 \\ 1 & 4 \\ 2 & 2 \end{pmatrix}. $$

(a) Compute the QR decomposition in **SymPy**. Give the shapes of $Q$ and $R$, and check $Q^TQ = I_2$ and $QR = A$.

(b) Compute the reduced and the complete QR in **NumPy**. Give the shapes and check `Q @ R` equals $A$ in both cases.

In [99]:
# Exercise 2


**Exercise 3 (SVD).** Let
$$ A = \begin{pmatrix} 2 & 2 \\ 1 & -1 \\ 2 & 2 \end{pmatrix}. $$

(a) In **SymPy**, compute $A^TA$ and its eigenvalues. Take square roots to get the singular values, and confirm with `A.singular_values()`.

(b) Compute the SVD with **NumPy**. Give the shapes of $U$, $\Sigma$, $V^T$, rebuild the $3\times 2$ matrix $\Sigma$, and check $U\Sigma V^T = A$.

In [100]:
# Exercise 3


